<a href="https://colab.research.google.com/github/irum-zahra-awan/geneai/blob/main/PromptEngineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Day 2: Advanced Prompt Engineering

## Session Objectives:
- Explore prompt tuning methods: Chain-of-Thought (CoT), Tree-of-Thought (ToT)
- Apply instruction tuning and function calling
- Understand system messages and few-shot learning

## Hands-On Activities:
1. Build ToT prompts using LangChain PromptTemplate
2. GPT-5 nano Prompt Refinement Exercises
3. Lab: Design & test assistant with system role instructions

## Setup and Installation

First, let's install the required packages and set up our environment.?

In [1]:
# Install required packages
# Run this cell first to install dependencies
!pip install -U langchain-core langchain-community langchain
!pip install openai langchain-openai python-dotenv


## Configuration

### Storing Secrets in Colab

1.  **Open the Secrets Panel:** In your Colab notebook, look for the 'key' icon (🔑) on the left sidebar. Click it to open the 'Secrets' panel.
2.  **Add a New Secret:** Click the '+ New secret' button.
3.  **Name your Secret:** In the 'Name' field, enter a name for your secret (e.g., `MY_API_KEY`). This is how you'll refer to it in your code.
4.  **Enter the Secret Value:** In the 'Value' field, paste or type your sensitive information (e.g., your API key).
5.  **Save:** Click 'Done'.
6.  **Enable Notebook Access:** Make sure the 'Notebook access' toggle next to your secret is enabled for the current notebook.

### Accessing Secrets in Code

Once you've stored your secret, you can access it in your Python code using `google.colab.userdata`:

In [2]:
# Import the userdata module
from google.colab import userdata

# Access your secret by its name
subscription_key = userdata.get('OPENAI_API_KEY_AZURE')
endpoint = userdata.get('ENDPOINT_AZURE')

# You can now use 'subscription_key' AND 'endpoint' in your code without exposing its value directly.

In [4]:
import os
from openai import AzureOpenAI

# from dotenv import load_dotenv
# load_dotenv()
# subscription_key=os.getenv("OPENAI_API_KEY")
# endpoint=os.getenv("endpoint")

model_name = "gpt-5-nano"
deployment = "gpt-5-nano"
api_version = "2024-12-01-preview"

client = AzureOpenAI(
    api_version=api_version,
    azure_endpoint=endpoint,
    api_key=subscription_key,
)

# Part 1: Chain-of-Thought (CoT) Prompting with LangChain

Chain-of-Thought helps the model reason step-by-step, improving accuracy for complex problems.

##### 1. WITHOUT CHAIN-OF-THOUGHT

In [17]:
# Without CoT
def solve_without_cot():
    """
    Solve a problem without Chain-of-Thought
    The model might jump to conclusions
    """
    prompt = """
A store has 15 apples. They sell 60% in the morning and then get a
shipment that doubles their remaining apples. How many apples do they have now?
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== WITHOUT CHAIN-OF-THOUGHT ===")
print(solve_without_cot())
print("\n" + "="*60 + "\n")

=== WITHOUT CHAIN-OF-THOUGHT ===

--- Token Usage ---
Input tokens (prompt): 42
Output tokens (completion): 319
Total tokens: 361
-------------------

12 apples

Explanation:
- 60% of 15 = 0.60 × 15 = 9 sold, so 15 − 9 = 6 left.
- Shipment doubles the remaining 6 → 6 × 2 = 12.




##### 2. WITH CHAIN-OF-THOUGHT

In [18]:
# With CoT
def solve_with_cot():
    """
    Solve the same problem with Chain-of-Thought prompting
    Explicitly ask for step-by-step reasoning
    """
    prompt = """
A store has 15 apples. They sell 60% in the morning and then get a
shipment that doubles their remaining apples. How many apples do they have now?

Let's solve this step-by-step:
1. First, calculate how many apples were sold
2. Then, find how many apples remain after the sale
3. Finally, calculate how many apples they have after the shipment doubles the remaining

Show your work for each step.
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[{"role": "user", "content": prompt}]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== WITH CHAIN-OF-THOUGHT ===")
print(solve_with_cot())
print("\n" + "="*60 + "\n")


=== WITH CHAIN-OF-THOUGHT ===

--- Token Usage ---
Input tokens (prompt): 97
Output tokens (completion): 498
Total tokens: 595
-------------------

Step 1: Calculate how many apples were sold
- 60% of 15 = 0.60 × 15 = 9 apples sold.

Step 2: Find how many apples remain after the sale
- 15 total − 9 sold = 6 apples remaining.

Step 3: Calculate how many apples they have after the shipment doubles the remaining
- Doubling the remaining 6 apples: 6 × 2 = 12 apples.

Answer: They have 12 apples now.




---

# Part 2: Tree-of-Thought (ToT) Prompting with LangChain

## What is Tree-of-Thought (ToT)?

Tree-of-Thought is an advanced prompting technique that:
- Generates multiple reasoning paths (branches)
- Evaluates each path
- Selects the best solution
- Useful for complex problem-solving that requires exploration of different approaches

### Comparison:
- **Chain-of-Thought (CoT)**: Linear reasoning (A → B → C → Solution)
- **Tree-of-Thought (ToT)**: Branching reasoning (explores multiple paths and selects best)

##### 1. Create a prompt template

In [20]:
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_openai import ChatOpenAI

# Step 1: Create a prompt template for generating multiple solution paths

# This template asks the LLM to think of different ways to solve the problem

tot_generation_template = PromptTemplate(
    input_variables=["numbers"],
    template="""
You are solving the Game of 24. You have these numbers: {numbers}
Goal: Use these 4 numbers and operations (+, -, *, /) to make 24.

Generate 3 DIFFERENT approaches to solve this problem.
For each approach, think step-by-step but don't solve completely yet.

Format your response as:
Approach 1: [describe the strategy]
Approach 2: [describe the strategy]
Approach 3: [describe the strategy]
"""
)

# Test the template
test_numbers = "4, 6, 8, 3"
prompt = tot_generation_template.format(numbers=test_numbers)

print("=== Generated Prompt ===")
print(prompt)
print("\n" + "="*50 + "\n")

=== Generated Prompt ===

You are solving the Game of 24. You have these numbers: 4, 6, 8, 3
Goal: Use these 4 numbers and operations (+, -, *, /) to make 24.

Generate 3 DIFFERENT approaches to solve this problem.
For each approach, think step-by-step but don't solve completely yet.

Format your response as:
Approach 1: [describe the strategy]
Approach 2: [describe the strategy]
Approach 3: [describe the strategy]





##### 2. Generate multiple approaches

In [21]:
# Step 2: Generate multiple approaches using GPT

def generate_tot_approaches(numbers):
    """
    Generate multiple approaches for solving the Game of 24

    Args:
        numbers: String of 4 numbers separated by commas

    Returns:
        String containing 3 different approaches
    """
    # Format the prompt with our numbers
    prompt = tot_generation_template.format(numbers=numbers)

    # Call GPT-4o to generate approaches
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=1  # Higher temperature for more creative approaches
    )

    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")

    return response.choices[0].message.content

# Generate approaches
approaches = generate_tot_approaches(test_numbers)
print("=== Generated Approaches ===")
print(approaches)
print("\n" + "="*50 + "\n")

KeyboardInterrupt: 

##### 3. Evaluate each approach

In [ ]:
# Step 3: Evaluate each approach

# Create evaluation prompt template
tot_evaluation_template = PromptTemplate(
    input_variables=["numbers", "approaches"],
    template="""
You are evaluating different approaches to solve the Game of 24 with numbers: {numbers}

Here are the proposed approaches:
{approaches}

For each approach:
1. Try to execute it step-by-step
2. Rate its likelihood of success (High/Medium/Low)
3. Explain why

Format:
Approach 1 Evaluation: [rating] - [explanation]
Approach 2 Evaluation: [rating] - [explanation]
Approach 3 Evaluation: [rating] - [explanation]
"""
)

def evaluate_tot_approaches(numbers, approaches):
    """
    Evaluate the generated approaches

    Args:
        numbers: String of 4 numbers
        approaches: String containing the approaches to evaluate

    Returns:
        String containing evaluation of each approach
    """
    prompt = tot_evaluation_template.format(numbers=numbers, approaches=approaches)

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=1  #0.3 Lower temperature for consistent evaluation
    )

    return response, response.choices[0].message.content

# Evaluate the approaches
response, evaluation = evaluate_tot_approaches(test_numbers, approaches)
print("=== Approach Evaluation ===")
print(evaluation)
print("\n" + "="*50 + "\n")

usage = response.usage
print(f"\n--- Token Usage ---")
print(f"Input tokens (prompt): {usage.prompt_tokens}")
print(f"Output tokens (completion): {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"-------------------\n")

##### 4. Select and execute the best approach

In [ ]:
# Step 4: Select and execute the best approach

tot_solution_template = PromptTemplate(
    input_variables=["numbers", "approaches", "evaluation"],
    template="""
You are solving the Game of 24 with numbers: {numbers}

Proposed approaches:
{approaches}

Evaluation:
{evaluation}

Based on the evaluation, select the BEST approach and solve the problem step-by-step.
Show your complete calculation to reach 24.

Format:
Selected Approach: [which one]
Step-by-step Solution:
[show all calculations]
Final Answer: [the equation that equals 24]
"""
)

def solve_with_best_approach(numbers, approaches, evaluation):
    """
    Select the best approach and solve the problem

    Args:
        numbers: String of 4 numbers
        approaches: String containing approaches
        evaluation: String containing evaluations

    Returns:
        String containing the final solution
    """
    prompt = tot_solution_template.format(
        numbers=numbers,
        approaches=approaches,
        evaluation=evaluation
    )

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ],
        temperature=1 #0.2  # Low temperature for accurate calculation
    )

    return response, response.choices[0].message.content

# Get the final solution
response, solution = solve_with_best_approach(test_numbers, approaches, evaluation)
print("=== Final Solution ===")
print(solution)
print("\n" + "="*50 + "\n")

usage = response.usage
print(f"\n--- Token Usage ---")
print(f"Input tokens (prompt): {usage.prompt_tokens}")
print(f"Output tokens (completion): {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"-------------------\n")


##### Complete ToT Pipeline Function

In [ ]:
# Step 5: Complete ToT Pipeline Function

def tree_of_thought_solver(numbers):
    """
    Complete Tree-of-Thought pipeline for Game of 24

    This function:
    1. Generates multiple approaches
    2. Evaluates each approach
    3. Selects and executes the best one

    Args:
        numbers: String of 4 numbers (e.g., "4, 6, 8, 3")

    Returns:
        Dictionary with approaches, evaluation, and solution
    """
    print(f"Solving Game of 24 for numbers: {numbers}")
    print("\nStep 1: Generating approaches...")

    # Generate approaches
    approaches = generate_tot_approaches(numbers)
    print("✓ Approaches generated\n")

    print("Step 2: Evaluating approaches...")
    # Evaluate approaches
    evaluation = evaluate_tot_approaches(numbers, approaches)
    print("✓ Evaluation complete\n")

    print("Step 3: Solving with best approach...")
    # Get solution
    solution = solve_with_best_approach(numbers, approaches, evaluation)
    print("✓ Solution found\n")

    return {
        "approaches": approaches,
        "evaluation": evaluation,
        "solution": solution
    }

# Test with different numbers
print("="*60)
print("TREE-OF-THOUGHT DEMONSTRATION")
print("="*60)

result = tree_of_thought_solver("3, 7, 8, 8")

print("\n" + "="*60)
print("FINAL SOLUTION:")
print("="*60)
print(result["solution"])

---

# Part 3: Prompt Refinement Exercises

## What is Prompt Refinement?

Prompt refinement is the iterative process of improving prompts to get better outputs. We'll practice:
1. Starting with a basic prompt
2. Identifying issues
3. Refining the prompt
4. Comparing results

## Exercise 3.1: From Basic to Advanced - Email Writing

##### 1. Basic Prompt

In [12]:
# Version 1: Basic Prompt (Vague)

def test_prompt_v1():
    """
    Test a basic, vague prompt
    Problem: Too generic, no context
    """
    prompt = "Write an email to a client."

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== VERSION 1: Basic Prompt ===")
print("Prompt: 'Write an email to a client.'")
print("\nOutput:")
result_v1 = test_prompt_v1()
print(result_v1)
print("\n" + "="*60 + "\n")

=== VERSION 1: Basic Prompt ===
Prompt: 'Write an email to a client.'

Output:

--- Token Usage ---
Input tokens (prompt): 13
Output tokens (completion): 1800
Total tokens: 1813
-------------------

Here are a few ready-to-use email templates you can customize. I’ve included a few common formats. If you share the purpose, tone, and some details, I can tailor one precisely.

Template 1: Project status update (formal)

Subject: Update on [Project/Order] – [Date]

Dear [Client Name],

I wanted to share a quick update on [Project/Order]. Here is our current status:

- Milestone reached: [milestone]
- Completed tasks: [tasks]
- In progress: [tasks]
- Next milestone/date: [date]

What we need from you: [any input or approvals required].
Proposed next steps: [brief plan and dates].

If you’d like, I can adjust the schedule or provide additional detail. Please let me know if you have any questions.

Best regards,
[Your Name]
[Title]
[Company]
[Phone]
[Email]

Template 2: Follow-up after a meet

##### 2. Adding Some Context

In [13]:
# Version 2: Adding Context

def test_prompt_v2():
    """
    Improved prompt with context
    Improvement: Added purpose and context
    """
    prompt = """
Write an email to a client informing them that their project delivery
will be delayed by 2 weeks due to technical challenges.
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== VERSION 2: Added Context ===")
print("Improvement: Added purpose and context")
print("\nOutput:")
result_v2 = test_prompt_v2()
print(result_v2)
print("\n" + "="*60 + "\n")

=== VERSION 2: Added Context ===
Improvement: Added purpose and context

Output:

--- Token Usage ---
Input tokens (prompt): 32
Output tokens (completion): 1569
Total tokens: 1601
-------------------

Subject: Update on [Project Name] Delivery Timeline

Dear [Client Name],

I hope you’re well. I’m writing to inform you that, despite our best efforts, we have encountered technical challenges that require additional time to ensure we meet our quality standards and your requirements. As a result, the project delivery will be delayed by two weeks from the original schedule.

Revised delivery timeline:
- Original delivery date: [Original Date]
- New delivery date: [New Date] (two weeks later)

What we’re doing to mitigate the delay:
- Focusing on root-cause analysis for the technical challenges, including [briefly describe areas, e.g., integration points, data pipelines, performance optimizations].
- Allocating additional resources to the critical path [e.g., add [X] engineers / specialists

##### 3. Adding Tone and Structure

In [14]:
# Version 3: Adding Tone and Structure

def test_prompt_v3():
    """
    Further improved prompt with tone and structure
    Improvement: Specified tone, format, and key points
    """
    prompt = """
Write a professional and empathetic email to a client informing them
that their project delivery will be delayed by 2 weeks due to technical challenges.

Tone: Professional but empathetic
Include:
- Brief explanation of the delay
- Reassurance about quality
- New timeline
- Offer of a brief call to discuss

Keep it concise (under 150 words).
"""

    response = client.chat.completions.create(
        model=deployment, #"gpt-4o",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")

    return response.choices[0].message.content

print("=== VERSION 3: Added Tone and Structure ===")
print("Improvement: Specified tone, format, and key points")
print("\nOutput:")
result_v3 = test_prompt_v3()
print(result_v3)
print("\n" + "="*60 + "\n")

=== VERSION 3: Added Tone and Structure ===
Improvement: Specified tone, format, and key points

Output:

--- Token Usage ---
Input tokens (prompt): 81
Output tokens (completion): 1488
Total tokens: 1569
-------------------

Subject: Update on project delivery timeline

Dear [Client Name],

I wanted to share a status update on your project. We’ve encountered technical challenges integrating a critical component, requiring additional validation to ensure reliability and performance. This will delay delivery by two weeks, but we’re prioritizing quality to prevent future rework.

New delivery date: February 22, 2026.

If you’d like to discuss the impact or adjust priorities, I’m available for a brief 15-minute call. Please share a convenient time, or I can propose options.

Thank you for your understanding and continued partnership.

Best regards,
[Your Name]
[Your Title]
[Company]




##### 4. Few-Shot Examples

In [15]:
# Version 4: Using LangChain PromptTemplate with Few-Shot Examples

def test_prompt_v4():
    """
    Best practice prompt using template and few-shot learning
    Improvement: Added examples and structured template
    """

    # Create a comprehensive prompt template
    email_template = PromptTemplate(
        input_variables=["situation", "delay_period", "reason"],
        template="""
You are a professional project manager writing to a valued client.

Situation: {situation}
Delay Period: {delay_period}
Reason: {reason}

Example of good client communication:
"Dear [Client], I wanted to reach out regarding [project]. Due to [brief reason],
we need to adjust our timeline by [period]. We're committed to delivering excellent
results and this additional time ensures we meet our quality standards. The new
delivery date is [date]. I'm available to discuss this at your convenience."

Write a professional email following this example's tone and structure.
Keep it under 150 words, empathetic, and solution-focused.
"""
    )

    # Fill in the template
    prompt = email_template.format(
        situation="project delivery delay notification",
        delay_period="2 weeks",
        reason="unexpected technical integration challenges"
    )

    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== VERSION 4: Template + Few-Shot Examples ===")
print("Improvement: Added examples and structured template")
print("\nOutput:")
result_v4 = test_prompt_v4()
print(result_v4)
print("\n" + "="*60 + "\n")

=== VERSION 4: Template + Few-Shot Examples ===
Improvement: Added examples and structured template

Output:

--- Token Usage ---
Input tokens (prompt): 139
Output tokens (completion): 2974
Total tokens: 3113
-------------------

Dear [Client Name], I wanted to reach out regarding [Project Name]. Due to unexpected technical integration challenges, we need to adjust our timeline by two weeks. We're committed to delivering excellent results, and this additional time ensures we meet our quality standards. The new delivery date is February 22, 2026. I understand this may impact your planning and appreciate your patience. I'm available to discuss this at your convenience.




---

# Part 4: System Role Instructions - Building Custom Assistants

## What are System Messages?

System messages:
- Define the assistant's behavior and personality
- Set guidelines for responses
- Persist across the entire conversation
- Are different from user messages

## Message Roles:
- **system**: Instructions for the assistant's behavior (set once)
- **user**: Messages from the user
- **assistant**: Responses from the AI

## Lab 4.1: Customer Support Assistant

In [22]:
# Create a customer support assistant with specific behavior

def create_support_assistant(user_message):
    """
    Customer support assistant with defined personality and rules

    Args:
        user_message: The customer's message

    Returns:
        Assistant's response
    """

    # Define the system message - this sets the assistant's behavior
    system_message = """
You are a friendly and professional customer support assistant for TechStore,
an online electronics retailer.

Your responsibilities:
- Help customers with order inquiries, returns, and technical questions
- Be empathetic and patient
- If you cannot solve an issue, offer to escalate to a human agent
- Always ask for order number when relevant
- Keep responses concise but helpful (under 100 words unless more detail is needed)

Tone: Friendly, professional, solution-oriented
Never: Make promises you can't keep, share customer data, or get defensive
"""

    # Make the API call with system and user messages
    response = client.chat.completions.create(
        model=deployment,
        messages=[
            {"role": "system", "content": system_message},
            {"role": "user", "content": user_message}
        ]
        #,temperature=0.7
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")

    return response.choices[0].message.content

# Test the assistant with different queries
test_queries = [
    "My package hasn't arrived yet and it's been 2 weeks!",
    "How do I return a defective laptop?",
    "Can you tell me about your warranty policy?"
]

print("=== CUSTOMER SUPPORT ASSISTANT DEMO ===")
print("\n")

for i, query in enumerate(test_queries, 1):
    print(f"Customer Query {i}: {query}")
    print(f"\nAssistant Response:")
    response = create_support_assistant(query)
    print(response)
    print("\n" + "="*60 + "\n")

=== CUSTOMER SUPPORT ASSISTANT DEMO ===


Customer Query 1: My package hasn't arrived yet and it's been 2 weeks!

Assistant Response:

--- Token Usage ---
Input tokens (prompt): 132
Output tokens (completion): 609
Total tokens: 741
-------------------

I’m sorry about the delay—that’s frustrating. Please share your TechStore order number so I can look up the tracking and check with the carrier. I’ll confirm what happened (delivery attempt, hold, or missing package) and advise the next steps (reship or refund if eligible). If you have the tracking number, feel free to include it as well. I can also escalate to a human agent if needed after I review.


Customer Query 2: How do I return a defective laptop?

Assistant Response:

--- Token Usage ---
Input tokens (prompt): 128
Output tokens (completion): 1404
Total tokens: 1532
-------------------

I’m sorry your laptop is defective—that’s frustrating. Please share your TechStore order number so I can pull up the details. You can also start 

## Lab 4.2: Multi-Turn Conversation with Memory

In [23]:
# Build a conversational assistant that remembers context

def chat_with_memory(conversation_history, new_user_message, system_prompt):
    """
    Chat function that maintains conversation history

    Args:
        conversation_history: List of previous messages
        new_user_message: The new message from user
        system_prompt: System instructions for the assistant

    Returns:
        Tuple of (assistant_response, updated_history)
    """

    # Add the new user message to history
    conversation_history.append({
        "role": "user",
        "content": new_user_message
    })

    # Create messages list: system message + conversation history
    messages = [
        {"role": "system", "content": system_prompt}
    ] + conversation_history

    # Get response from GPT-4o
    response = client.chat.completions.create(
        model=deployment,
        messages=messages,
        #temperature=0.7
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    assistant_message = response.choices[0].message.content

    # Add assistant's response to history
    conversation_history.append({
        "role": "assistant",
        "content": assistant_message
    })

    return assistant_message, conversation_history

# Example: Personal Tutor Assistant
tutor_system_prompt = """
You are a patient and encouraging Python programming tutor.

Your approach:
- Ask questions to understand the student's level
- Explain concepts with simple examples
- Encourage practice and experimentation
- Remember what the student has learned in this conversation
- Adjust difficulty based on student's responses

Teaching style: Socratic method - ask guiding questions rather than giving direct answers
"""

# Initialize conversation
history = []

print("=== PYTHON TUTOR CONVERSATION DEMO ===")
print("(Notice how the assistant remembers context across messages)\n")

# Message 1
print("Student: I want to learn about Python lists")
response1, history = chat_with_memory(
    history,
    "I want to learn about Python lists",
    tutor_system_prompt
)
print(f"Tutor: {response1}\n")
print("="*60 + "\n")

# Message 2 - references previous context
print("Student: I'm a complete beginner")
response2, history = chat_with_memory(
    history,
    "I'm a complete beginner",
    tutor_system_prompt
)
print(f"Tutor: {response2}\n")
print("="*60 + "\n")

# Message 3 - builds on conversation
print("Student: Can you give me an example?")
response3, history = chat_with_memory(
    history,
    "Can you give me an example?",
    tutor_system_prompt
)
print(f"Tutor: {response3}\n")
print("="*60 + "\n")

print(f"\nConversation length: {len(history)} messages")

=== PYTHON TUTOR CONVERSATION DEMO ===
(Notice how the assistant remembers context across messages)

Student: I want to learn about Python lists

--- Token Usage ---
Input tokens (prompt): 89
Output tokens (completion): 2018
Total tokens: 2107
-------------------

Tutor: Awesome! Lists are a great place to start with Python. Before we dive in, a couple quick questions to tailor things:

- Have you used Python before, or is this your first time?
- Do you want examples with numbers, with text, or a mix?
- Would you prefer a step-by-step walk-through or short hands-on exercises you can try on your own?

If you’re not sure, that’s fine—we’ll start simple and build up.

Here’s a gentle, starting overview with small examples.

What a list is
- A list is an ordered collection of items. It’s mutable, meaning you can change it after you create it.
- You make a list with square brackets: my_list = [1, 2, 3] or with strings: fruits = ["apple", "banana", "cherry"]

Basic operations (quick ideas)
-

## Lab 4.3: Building Different Assistant Personalities

In [24]:
# Compare different system prompts for the same query

# user question
user_question = "Explain what machine learning is"

# Different system prompts create different personalities

# Assistant 1: Formal Academic
academic_system = """
You are a university professor specializing in computer science.
Use precise, academic language with technical terminology.
Structure explanations formally with clear definitions.
"""

# Assistant 2: Casual Explainer
casual_system = """
You are a friendly tech enthusiast explaining concepts to a friend.
Use simple language, everyday analogies, and a conversational tone.
Make complex topics feel approachable and fun.
"""

# Assistant 3: Socratic Teacher
socratic_system = """
You are a teacher who uses the Socratic method.
Instead of explaining directly, ask thought-provoking questions.
Guide the student to discover the answer themselves.
"""

def get_response_with_personality(system_prompt, user_msg):
    """Get response with specific personality"""
    response = client.chat.completions.create(
        model= deployment,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_msg}
        ],
        #temperature=0.7
    )
    usage = response.usage
    print(f"\n--- Token Usage ---")
    print(f"Input tokens (prompt): {usage.prompt_tokens}")
    print(f"Output tokens (completion): {usage.completion_tokens}")
    print(f"Total tokens: {usage.total_tokens}")
    print(f"-------------------\n")
    return response.choices[0].message.content

print("=== SAME QUESTION, DIFFERENT PERSONALITIES ===")
print(f"\nUser Question: {user_question}\n")
print("="*60)

# Get responses from each personality
print("\n1. FORMAL ACADEMIC RESPONSE:")
print("="*60)
print(get_response_with_personality(academic_system, user_question))

print("\n\n2. CASUAL EXPLAINER RESPONSE:")
print("="*60)
print(get_response_with_personality(casual_system, user_question))

print("\n\n3. SOCRATIC TEACHER RESPONSE:")
print("="*60)
print(get_response_with_personality(socratic_system, user_question))

=== SAME QUESTION, DIFFERENT PERSONALITIES ===

User Question: Explain what machine learning is


1. FORMAL ACADEMIC RESPONSE:

--- Token Usage ---
Input tokens (prompt): 42
Output tokens (completion): 2450
Total tokens: 2492
-------------------

Definition (high level)
- Machine learning is the study and application of algorithms that improve their performance on a specified task through experience, where experience is provided by data, rather than being explicitly programmed with task-specific rules.

Formal framing
- Let X denote the input space and Y denote the output space (labels, targets, or responses).
- Suppose data are drawn i.i.d. from an unknown joint distribution P(X, Y).
- A learning problem provides a hypothesis class H, where each h ∈ H is a predictor h: X → Y, and a loss function ℓ: Y × Y → ℝ+ that measures the penalty for predicting h(x) when the true outcome is y.
- The training data D = { (x_i, y_i) }_{i=1}^n are used to select a predictor by solving an optimization

## Lab 4.4: Advanced System Prompt - JSON Output Format

System prompts can also enforce output formatting, such as JSON.

In [25]:
# System prompt that enforces JSON output

import json

json_system_prompt = """
You are a data extraction assistant.
Extract information from text and return it in valid JSON format.

For restaurant reviews, extract:
- restaurant_name (string)
- rating (number 1-5)
- cuisine_type (string)
- price_range (string: "$", "$$", "$$$", or "$$$$")
- key_highlights (list of strings)
- would_recommend (boolean)

Always return ONLY valid JSON, no additional text.
"""

# Test with a restaurant review
review_text = """
I visited The Golden Spoon last night and it was amazing! This Italian restaurant
has the best homemade pasta I've ever had. The prices are a bit high - we spent
about $80 per person - but the quality is worth it. The ambiance is romantic and
the service was excellent. I'd definitely go back. I'd give it 5 stars!
"""

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "system", "content": json_system_prompt},
        {"role": "user", "content": review_text}
    ]
    #,temperature=0.3  # Low temperature for consistent formatting
)

# Get the response
json_output = response.choices[0].message.content

usage = response.usage
print(f"\n--- Token Usage ---")
print(f"Input tokens (prompt): {usage.prompt_tokens}")
print(f"Output tokens (completion): {usage.completion_tokens}")
print(f"Total tokens: {usage.total_tokens}")
print(f"-------------------\n")

print("=== STRUCTURED JSON OUTPUT ===")
print("\nRaw Output:")
print(json_output)

# Parse and pretty-print the JSON
print("\n" + "="*60)
print("Parsed JSON:")
try:
    parsed_data = json.loads(json_output)
    print(json.dumps(parsed_data, indent=2))

    # You can now use this data programmatically
    print("\n" + "="*60)
    print("Accessing data:")
    print(f"Restaurant: {parsed_data['restaurant_name']}")
    print(f"Rating: {parsed_data['rating']}/5")
    print(f"Would recommend: {parsed_data['would_recommend']}")

except json.JSONDecodeError:
    print("Error: Output is not valid JSON")


--- Token Usage ---
Input tokens (prompt): 172
Output tokens (completion): 541
Total tokens: 713
-------------------

=== STRUCTURED JSON OUTPUT ===

Raw Output:
{
  "restaurant_name": "The Golden Spoon",
  "rating": 5,
  "cuisine_type": "Italian",
  "price_range": "$$$",
  "key_highlights": [
    "homemade pasta",
    "romantic ambiance",
    "excellent service",
    "high-quality ingredients",
    "worth the price"
  ],
  "would_recommend": true
}

Parsed JSON:
{
  "restaurant_name": "The Golden Spoon",
  "rating": 5,
  "cuisine_type": "Italian",
  "price_range": "$$$",
  "key_highlights": [
    "homemade pasta",
    "romantic ambiance",
    "excellent service",
    "high-quality ingredients",
    "worth the price"
  ],
  "would_recommend": true
}

Accessing data:
Restaurant: The Golden Spoon
Rating: 5/5
Would recommend: True
